Evaluate Neural Networks on Data
===

In the `4_nerual_likelihood_ratio_estimation.ipynb` and `5_systematic_uncertainty_training.ipynb` notebook, we trained neural networks to estimate the relevant density ratios needed to build our full statistical model. In this notebook, we evaluate density ratios using those trained models for the Asimov dataset. 

The full model is written as

$$\frac{p(x|\mu, \alpha)}{p_{ref}(x)} = \frac{1}{\sum_c G_c(\alpha) \cdot f_c(\mu) \cdot \nu_c} \sum_c f_c(\mu) \cdot G_c(\alpha)\cdot \nu_c \cdot g_c(x|\alpha) \cdot \underbrace{\frac{p_c\left(x\right)}{p_{ref}(x)}}_{NN}$$
where,
$$g_c(x|\alpha) = \frac{p_c(x|\alpha)}{p_\text{c}(x)} = \prod_p \frac{p_c(x|\alpha_p)}{p_\text{c}(x)}$$

and we used a HistFactory-style interpolation generalized to a per-event formulation:

$$\frac{p_c(x|\alpha_p^{eval})}{p_\text{c}(x)} = \begin{cases}
    \bigg(\underbrace{\frac{p_c(x|\alpha_p^+)}{p_\text{c}(x)}}_{NN}\bigg)^{\alpha_p^\text{eval}}& \alpha_p^\text{eval}>1\\
    1+\sum_{n=1}^6c_n\cdot (\alpha_p^\text{eval})^n& -1\leq\alpha_p^\text{eval}\leq 1\\
    \bigg(\underbrace{\frac{p_c(x|\alpha_p^-)}{p_\text{c}(x)}}_{NN}\bigg)^{-\alpha_p^\text{eval}}& \alpha_p^\text{eval}<-1\\
    \end{cases}$$


**These are the density ratios that we will evaluate in this notebook.**

In [1]:
import os
import sys, pathlib
from pathlib import Path
import argparse
import warnings
import logging
import numpy as np
import yaml
import mplhep as hep
import pickle

import nsbi_common_utils

warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

hep.style.use(hep.style.ATLAS)

/home/jsandesara_umass_edu/NSBI-workflow-tutorial/src/nsbi_common_utils/plotting.py:10: FutureWarning: ``set_style`` is deprecated: Naming convention is changing to match mpl. Use ``mplhep.style.use()``.
  hep.set_style("ATLAS")


In [2]:
def load_config(path):
    with open(path, "r") as f:
        return yaml.safe_load(f)


In [3]:
workflow_config_path = "./config.pipeline.yaml"

In [4]:
# Load the workflow parameters
config_workflow_nominal         = load_config(workflow_config_path)["neural_likelihood_ratio_estimation"]
config_workflow_systematics     = load_config(workflow_config_path)["systematic_uncertainty"]

nsbi_fit_config_path = config_workflow_nominal["nsbi_fit_config"]

print(f"Initializing NSBI ConfigManager from: {nsbi_fit_config_path}")
fit_config_nsbi = nsbi_common_utils.configuration.ConfigManager(file_path_string=nsbi_fit_config_path)


Initializing NSBI ConfigManager from: ./config_fit_nsbi.yml


In [5]:
# Input features for training - these features should be the same ones used for nominal training
features, features_scaling = fit_config_nsbi.get_training_features()

print(f"Training features used = \n\n {features}")

Training features used = 

 ['DER_mass_transverse_met_lep', 'log_DER_mass_vis', 'log_DER_pt_h', 'DER_deltar_had_lep', 'log_DER_pt_tot', 'log_DER_sum_pt', 'DER_pt_ratio_lep_had', 'DER_met_phi_centrality']


In [6]:
branches_to_load = features + ['presel_score']

datasets_helper = nsbi_common_utils.datasets.datasets(
    config_path=nsbi_fit_config_path,
    branches_to_load=branches_to_load
)


# Load the full dataset, including systematic variations defined in the config file
dataset_incl_dict = datasets_helper.load_datasets_from_config(load_systematics=False)

In [7]:
# Get the nominal sample
dataset_nominal = dataset_incl_dict["Nominal"].copy()

In [8]:
# Filter out events to get the NSBI signal region 
region = config_workflow_nominal["filter_region"]

print(f"Filtering datasets for region: {region}")
# dataset_SR_dict = datasets_helper.filter_region_by_type(dataset_incl_dict, region = region)
dataset_SR = datasets_helper.filter_region_dataset(dataset_nominal, region=region)

Filtering datasets for region: SR


In [27]:
path_to_saved_data = config_workflow_nominal["saved_data_path"]

The Asimov Dataset
--

In [29]:
# Get the anchor/basis points used to build the full statistical model
basis_processes = fit_config_nsbi.get_basis_samples()
print(f"Basis processes: {basis_processes}")

print("Merging dataframes for final evaluation.")
dataset_Asimov_SR = datasets_helper.merge_dataframe_dict_for_training(
    dataset_SR, None, samples_to_merge=basis_processes
)

Basis processes: ['ztautau', 'ttbar', 'htautau']
Merging dataframes for final evaluation.


In [30]:
# Save Asimov weights for inference
weight_save_path = fit_config_nsbi.get_channel_asimov_weight_path(channel_name=region) # Get path to asimov weights from fit config
np.save(weight_save_path, dataset_Asimov_SR.weights.to_numpy()) # save the weights

Nominal density ratio evaluation
===

We evaluate the trained NNs to predict

$$\frac{p_c}{p_{ref}}(x)$$

for each process $p_c$ of the statistical model

In [31]:
# Get the path where trained models were saved
training_input_dir_name = config_workflow_nominal["output_training_dir"]
trained_models_path = os.path.join(path_to_saved_data, training_input_dir_name)

In [32]:
ensemble_members    = config_workflow_nominal.get("num_ensemble_members_evaluation", 1)
aggregation_type    = config_workflow_nominal.get("ensemble_aggregation_type", "mean_ratio")

print(f"Evaluating on Asimov with a total of {ensemble_members} NNs for each process.")
print(f"Aggregating the ensemble output form {ensemble_members} NNs using {aggregation_type} strategy.")

In [33]:
for process_type in basis_processes:

    path_to_saving_evaluated_ratios         = os.path.join(trained_models_path, f'output_ratios_{process_type}/')

    score_pred = np.ones((ensemble_members, dataset_Asimov_SR.shape[0]))
    ratio_pred = np.ones((ensemble_members, dataset_Asimov_SR.shape[0]))
    loaded_indices = []

    for ensemble_index in range(ensemble_members):

        path_to_trained_models                  = os.path.join(trained_models_path, f'output_model_params_{process_type}{ensemble_index}/')

        path_to_saved_scaler        = f"{path_to_trained_models}model_scaler{ensemble_index}.bin"
        path_to_saved_model         = f"{path_to_trained_models}model{ensemble_index}.onnx"

        model_file = Path(path_to_saved_model)
        if not model_file.is_file():
            print(f"No model exists for ensemble index {ensemble_index} for process {process_type}")
            continue
        print(f"Reading saved models from {path_to_saved_model}")

        scaler, model_NN                = nsbi_common_utils.training.load_trained_model(path_to_saved_model, path_to_saved_scaler)
        score_pred[ensemble_index]      = nsbi_common_utils.training.predict_with_model(dataset_Asimov_SR[features], scaler, model_NN)
        ratio_pred[ensemble_index]      = nsbi_common_utils.training.convert_score_to_ratio(score_pred[ensemble_index])    
        loaded_indices.append(ensemble_index)

    if aggregation_type == 'median_ratio':
        ratio_ensemble = np.median(ratio_pred[loaded_indices], axis=0)
        
    elif aggregation_type == 'mean_ratio':
        ratio_ensemble = np.mean(ratio_pred[loaded_indices], axis=0)
        
    elif aggregation_type == 'median_score':
        score_aggregate = np.median(score_pred[loaded_indices], axis=0)
        ratio_ensemble = score_aggregate / (1.0 - score_aggregate)
        
    elif aggregation_type == 'mean_score':
        score_aggregate = np.mean(score_pred[loaded_indices], axis=0)
        ratio_ensemble = score_aggregate / (1.0 - score_aggregate)

    else:
        raise Exception("aggregation_type not recognized, please choose between median_ratio, mean_ratio, median_score or mean_score")

    saved_ratio_path = f"{path_to_saving_evaluated_ratios}ratio_{process_type}.npy"
    os.makedirs(path_to_saving_evaluated_ratios, exist_ok=True)
    np.save(saved_ratio_path, ratio_ensemble)

Evaluating and saving nominal density ratios on Asimov dataset
Reading saved models from saved_datasets/output_training_nominal/output_model_params_ztautau0/model0.onnx
Reading saved models from saved_datasets/output_training_nominal/output_model_params_ztautau1/model1.onnx
Reading saved models from saved_datasets/output_training_nominal/output_model_params_ztautau2/model2.onnx
Reading saved models from saved_datasets/output_training_nominal/output_model_params_ztautau3/model3.onnx
Reading saved models from saved_datasets/output_training_nominal/output_model_params_ztautau4/model4.onnx
Reading saved models from saved_datasets/output_training_nominal/output_model_params_ztautau5/model5.onnx
Reading saved models from saved_datasets/output_training_nominal/output_model_params_ztautau6/model6.onnx
Reading saved models from saved_datasets/output_training_nominal/output_model_params_ztautau7/model7.onnx
Reading saved models from saved_datasets/output_training_nominal/output_model_params_ztau

Systematics density ratio evaluation
===

We evaluate the trained NNs to predict two ratios

$$\frac{p_c(x|\alpha_p^+)}{p_\text{c}(x)} \, , \frac{p_c(x|\alpha_p^-)}{p_\text{c}(x)},$$

for each nuisance parameter $\alpha_p$, for each process $p_c$ that is affected by the corresponding nuisance parameter.

In [35]:
print("Running evaluation on Asimov with systematic variation networks")

# Get the path where trained models were saved
training_input_dir_name = config_workflow_systematics["output_training_dir"]
trained_models_path = os.path.join(path_to_saved_data, training_input_dir_name)

print(f"Trained models path: {trained_models_path}")

Running evaluation on Asimov with systematic variation networks
Trained models path: saved_datasets/output_training_systematics


In [36]:
calibration_flag        = config_workflow_systematics["training_settings"].get("calibration", False)

for process_type in basis_processes:

    ensemble_index = '' #TODO: support ensemble evaluations for systematics too

    for dict_syst in fit_config_nsbi.config["Systematics"]:

        # Only evaluate norm+shape systematics where the process_type is involved
        if (process_type not in dict_syst["Samples"]) or (dict_syst["Type"] != "NormPlusShape"): continue

        syst = dict_syst["Name"]

        for direction in ["Up", "Dn"]:

            output_name = f'{process_type}_{syst}_{direction}'
                    
            path_to_saving_evaluated_ratios     = os.path.join(trained_models_path, f'output_ratios_{output_name}/')
            path_to_trained_models              = os.path.join(trained_models_path, f'output_model_params_{output_name}/')

            path_to_saved_scaler        = f"{path_to_trained_models}model_scaler{ensemble_index}.bin"
            path_to_saved_model         = f"{path_to_trained_models}model{ensemble_index}.onnx"

            print(f"Evaluating and Saving Ratios for {process_type} {syst} {direction}")

            print(f"Reading saved models from {path_to_trained_models}")
            scaler, model_NN                = nsbi_common_utils.training.load_trained_model(path_to_saved_model, path_to_saved_scaler)

            path_to_calibrator_model    = None
            if calibration_flag:
                path_to_calibrator_model         = f"{path_to_trained_models}model_calibrated_hist{ensemble_index}.obj"
                if not os.path.exists(path_to_calibrator_model):
                    logger.warning(f"No calibration model found with name {path_to_calibrator_model}")
                    calibration_model = None
                else:
                    file_calibration = open(path_to_calibrator_model, 'rb') 
                    calibration_model = pickle.load(file_calibration)
            else:
                calibration_model = None

            score_pred      = nsbi_common_utils.training.predict_with_model(dataset_Asimov_SR[features], 
                                                                                           scaler, model_NN, 
                                                                                           calibration_model = calibration_model)
            ratio_pred      = nsbi_common_utils.training.convert_score_to_ratio(score_pred)    

            saved_ratio_path = f"{path_to_saving_evaluated_ratios}ratio_{process_type}.npy"

            os.makedirs(path_to_saving_evaluated_ratios, exist_ok=True)
            
            np.save(saved_ratio_path, ratio_pred)

            print(f"Systematic density ratios for {syst}_{direction} affecting the {process_type} basis point saved to: {saved_ratio_path}")

            print("All systematic density ratios evaluated on Asimov and saved.")

Evaluating and Saving Ratios for ztautau JES Up
Reading saved models from saved_datasets/output_training_systematics/output_model_params_ztautau_JES_Up/
Systematic density ratios for JES_Up affecting the ztautau basis point saved to: saved_datasets/output_training_systematics/output_ratios_ztautau_JES_Up/ratio_ztautau.npy
All systematic density ratios evaluated on Asimov and saved.
Evaluating and Saving Ratios for ztautau JES Dn
Reading saved models from saved_datasets/output_training_systematics/output_model_params_ztautau_JES_Dn/
Systematic density ratios for JES_Dn affecting the ztautau basis point saved to: saved_datasets/output_training_systematics/output_ratios_ztautau_JES_Dn/ratio_ztautau.npy
All systematic density ratios evaluated on Asimov and saved.
Evaluating and Saving Ratios for ztautau TES Up
Reading saved models from saved_datasets/output_training_systematics/output_model_params_ztautau_TES_Up/
Systematic density ratios for TES_Up affecting the ztautau basis point saved 